In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


In this notebook I want to plot examples of regional effects, and plot EVERY single individual example of electrode or pair effects so that I can later look through them (and log them) for selection in the paper.

Note: The code to run PAC is plotting examples already, but it's plotting for all retrieval vs. all baseline. This is not what I will want for the paper (I want to separate by RPE/memory, etc)

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import mne
import mne_connectivity
from mne_connectivity import phase_slope_index, seed_target_indices, spectral_connectivity_epochs
from glob import glob
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from scipy.stats import zscore, linregress, ttest_ind, ttest_rel, ttest_1samp
from scipy.ndimage import uniform_filter1d
import pandas as pd
from mne.preprocessing.bads import _find_outliers
import fooof
# Import plotting functions
from fooof.plts.spectra import plot_spectrum, plot_spectra
# Import the FOOOF object
from fooof import FOOOF
from fooof import FOOOFGroup
import os 
import joblib
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pickle
from tqdm import tqdm
from IPython.display import clear_output
import warnings 

# I only want to see warnings once
warnings.filterwarnings('ignore')

from joblib import delayed, Parallel
from statsmodels.stats import multitest
import re

plot_dir = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit_PlotExamples'
os.makedirs(plot_dir, exist_ok=True)


In [ ]:
import sys
sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')


In [ ]:
from membandit.plotting_style import set_plot_style
set_plot_style()

## First, get the behavior data

In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site!='UI']
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]


days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')



In [ ]:
combined_df['reward'] = combined_df.rpe.apply(lambda x: 1 if x > 0 else 0)

In [ ]:
learn_df['gamble_rt_log'] = np.log(learn_df.gamble_rt)

combined_df['gamble_rt_log'] = np.log(combined_df.gamble_rt_gamble)
combined_df['recog_rt_log'] = np.log(combined_df.recog_rt)

In [ ]:
combined_df['memory_outcome'] = combined_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome') 

# Make supplementary tables

## First, need a table of all the patients, their demographic information, and their electrode counts/contributions

In [ ]:
roi = ['dmPFC', 'dlPFC', 'OFC', 'ACC', 'AMY', 'HPC']

paper_df = subj_df[['MSSMCode', 'Age', 'Sex']]
for region in roi: 
    paper_df[region] = np.nan

for subj_id in paper_df.MSSMCode.unique():
    # load the electrode data for each person
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
    elecs_in_roi = elec_df[elec_df.salman_region.isin(roi)].salman_region.value_counts()
    for region in elecs_in_roi.index:
        paper_df.loc[(paper_df.MSSMCode==subj_id), region] = elecs_in_roi[region]

In [ ]:
paper_df

# Plot filtered LFP data

In [ ]:
roi = ['dmPFC', 'dlPFC', 'OFC', 'ACC', 'AMY', 'HPC']
boi = 'high_theta'
foi = [2, 200]
freqs = np.logspace(*np.log10([2, 200]), num=30)

In [ ]:
hpc_c = np.array([0.44314, 0.2902, 1])
amy_c = np.array([0.49412, 0, 0.90196])
ofc_c = np.array([1, 0.75686, 0.41569])
acc_c = np.array([0.7098, 0.27059, 0.68235])
dmpfc_c = np.array([0.40784, 0.01961, 0])
dlpfc_c = np.array([1, 0.06275, 0.06275])

In [ ]:
# Let's load the power epochs for specific conditions: 

analysis_evs = ['feedback_start']
day = 'Day1'

# tfr_group_data = {f'{a}': {f'{b}': [] for b in conditions} for a in rois}

for subj_id in tqdm(['MS028']): 
    for event in analysis_evs:
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        
        # Load the data 
        save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
        event_epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif', preload=True)
        
        # Get electrode df 
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
               
        # 1) Immediately crop this to the roi: 
        elecs_in_roi = elec_df[elec_df.salman_region.isin(roi)].label.values.tolist()
        rois_per_chan = elec_df[elec_df.salman_region.isin(roi)].salman_region.values.tolist()
        event_epochs_reref.pick(elecs_in_roi)
        
        filter_data = event_epochs_reref.filter(foi[0], foi[1])
        filter_data.crop(tmin=0, tmax=1.5)

#         # 4) Arrange this data into a pandas dataframe 
        
#         # Reshape the array into a shape that can be converted into DataFrame
#         roi_df = [] 
        for ix, label in enumerate(elecs_in_roi):
            filt_lfp = filter_data._data[:,ix,:]
            region_for_chan = rois_per_chan[ix]

            if region_for_chan=='OFC':
                color = ofc_c
            elif region_for_chan=='HPC': 
                color = hpc_c
            elif region_for_chan=='AMY':
                color = amy_c
            elif region_for_chan=='dmPFC':
                color = dmpfc_c
            elif region_for_chan=='dlPFC':
                color = dlpfc_c
            elif region_for_chan=='ACC':
                color = acc_c

            # Let's plot the first trial of each one
            fig, ax = plt.subplots(1, 1, figsize=(10, 4), dpi=300)
            ax.plot(filt_lfp[0, :], color=color)
            plt.savefig(f'{plot_dir}/{event}_{subj_id}_{region_for_chan}_{label}_trial0_rawLFP.pdf', dpi=300)




#             trial_col = np.repeat(np.arange(filter_data._data.shape[0]) + 1, filt_lfp.shape[0]/filter_data._data.shape[0])
#             times_col = list(filter_data.times) * filter_data._data[:,ix,:].shape[0]

#             # Create DataFrame
#             df = pd.DataFrame({
#                 'trials': trial_col,
#                 'label': label,
#                 'time': times_col,
#                 'filtered_lfp': filt_lfp,
#                 'band': boi,
#                 'region': elec_df[elec_df.label==label].salman_region.values[0]

#             })
            
#             # Smooth data, if need be:
            
#             roi_df.append(df)

#         roi_df = pd.concat(roi_df).reset_index(drop=True)
            
# # # #         # Filter out the nans
# # # #         df.dropna(inplace=True)
                    
#         if event in ['cue_on', 'recog_time']:
#             trials_to_include = combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_mem').reset_index(drop=True).trials_mem.values.astype(int)
#         else:
# # # #         # 5) Merge with the appropriate behavioral dataframe (depending on condition)
#             # trials_to_include = learn_df[(learn_df.participant==subj_id)].trials.values - 1
                
#             trials_to_include = combined_df[(combined_df.condition == 'Day1') & (combined_df.participant == subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble']).reset_index(drop=True).trials_gamble.values.astype(int) - 1

#         roi_df = roi_df[roi_df['trials'].isin(trials_to_include)]
        
#         # 
#         subj_df = combined_df[(combined_df.condition == 'Day1') & (combined_df.participant == subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble']).reset_index(drop=True)
#         subj_df['trials'] = subj_df['trials_gamble']
# #         subj_df.rename(columns={'trials_gamble':'trial'}, inplace=True)
#         subj_df['cat_zrpe'] = pd.qcut(subj_df.rpe, 2, labels = ['low', 'high'])
#         subj_df['cat_zpm'] = pd.qcut(subj_df.DPRIME, 2, labels = ['low', 'high'])
# #         subj_df = subj_df[['trial', 'zrpe', 'zpm', 'cat_zrpe', 'cat_zpm']] 

#         merged_df = roi_df.merge(subj_df, on=['trials'])
        
#         for label in merged_df.label.unique():
#             fig, ax = plt.subplots(1, 1, figsize=(10, 4), dpi=300)
#             sns.lineplot(data=merged_df[(merged_df.label==label) ], x='time', y='filtered_lfp', 
#                          hue='hits', errorbar='se', ax=ax, hue_order=[False, True], ci=None)
#             sns.despine()
#             ax.set(ylabel = 'power (z)', xlabel = 'time (s)', title=f'{label}')

#             plt.savefig(f'{base_dir}/work/qasims01/MemoryBanditData/EMU/plots/{event}_{elec_df[elec_df.label==label].salman_region.values[0]}_{boi}_RPE_{subj_id}_{label}_rawLFP.pdf', dpi=300)
# #             plt.close()


In [ ]:
f'{base_dir}/work/qasims01/MemoryBanditData/EMU/plots'

In [ ]:
filter_data._data.shape

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4), dpi=300)
ax.plot(filter_data._data[0, 2, :])

In [ ]:
elecs_in_roi

In [ ]:
elecs_in_roi

In [ ]:
merged_df.hits

In [ ]:
merged_df.label

In [ ]:
merged_df.label.unique()

In [ ]:
sns.lineplot(data=df[(df.trials==1) & (df.time<0.1)], x='time', y='filtered_lfp')
sns.lineplot(data=df[(df.trials==2) & (df.time<0.1)], x='time', y='filtered_lfp')


In [ ]:
filter_data._data.shape

# Encoding power changes in HPC and OFC 

## First, HFA increases in HPC for positive RPEs 

In [ ]:
# First, let's load the TFR data: 

In [ ]:
base_dir = '/sc/arion' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv') 

In [ ]:
roi = 'OFC'
boi = 'hfa'
foi = [70, 200]
freqs = np.logspace(*np.log10([2, 200]), num=30)

In [ ]:
# Let's load the power epochs for specific conditions: 

analysis_evs = ['feedback_start']

# tfr_group_data = {f'{a}': {f'{b}': [] for b in conditions} for a in rois}

for subj_id in tqdm(subj_ids): 
    for event in analysis_evs:
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        
        # Load the data 
        full_tfr = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
        
        # Get electrode df 
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
        
        
        # 1) Immediately crop this to the roi: 
        elecs_in_roi = elec_df[elec_df.salman_region==roi].label.values.tolist()
        
        nchans = len(elecs_in_roi)
        
        if nchans == 0: 
            print(f'no electrodes in {subj_id}')
            continue
        full_tfr.pick(elecs_in_roi)
        
        
        
        # 2) Immediately restrict this to the frequency of interest and average it within band 
        freqs_in_boi = (freqs>foi[0]) & (freqs<foi[-1])
        freq_avged_data = np.squeeze(full_tfr._data[:, :, freqs_in_boi, :].mean(axis=2))
        if nchans == 1: 
            # expand freq_avged_data:
            freq_avged_data = np.expand_dims(freq_avged_data, axis=1)
        
#         # 3) Smooth data, if need be:
        if boi == 'hfa':
            # Assuming arr is your numpy array of shape (80, 3, 751)
            smoothed_arr = np.zeros_like(freq_avged_data)
            for trial in range(smoothed_arr.shape[0]):
                for chan in range(smoothed_arr.shape[1]):
                    smoothed_arr[trial, chan, :] = pd.Series(freq_avged_data[trial, chan, :]).rolling(window=200, min_periods=1, step=1, closed='both').apply(lambda x: np.nanmean(x)).values
                
        else: 
            smoothed_arr = freq_avged_data
        
        # 4) Arrange this data into a pandas dataframe 
        
        # Reshape the array into a shape that can be converted into DataFrame
        roi_df = [] 
        for ix, label in enumerate(elecs_in_roi):
            if boi == 'hfa':
                flat_pow = smoothed_arr[:,ix,:].flatten()
            else:
                flat_pow = freq_avged_data[:,ix,:].flatten()
            trial_col = np.repeat(np.arange(freq_avged_data.shape[0]) + 1, flat_pow.shape[0]/freq_avged_data.shape[0])
            times_col = list(full_tfr.times) * smoothed_arr[:,ix,:].shape[0]

            # Create DataFrame
            df = pd.DataFrame({
                'trials': trial_col,
                'label': label,
                'time': times_col,
                'smoothed_power': flat_pow,
                'band': boi,
                'region': roi
            })
            
            # Smooth data, if need be:
            
            roi_df.append(df)

        roi_df = pd.concat(roi_df).reset_index(drop=True)
            
# # #         # Filter out the nans
# # #         df.dropna(inplace=True)
                    
        if event in ['cue_on', 'recog_time']:
            trials_to_include = combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_gamble').reset_index(drop=True).trials_gamble.values.astype(int)
        else:
# # #         # 5) Merge with the appropriate behavioral dataframe (depending on condition)
            trials_to_include = learn_df[(learn_df.participant==subj_id)].trials.values - 1
        roi_df = roi_df[roi_df['trials'].isin(trials_to_include)]
        
        # 
        subj_df = learn_df[(learn_df.participant==subj_id)].sort_values(by='trials')
#         subj_df.rename(columns={'trials_gamble':'trial'}, inplace=True)
        subj_df['cat_zrpe'] = pd.qcut(subj_df.rpe, 2, labels = ['low', 'high'])
        subj_df['cat_zpm'] = pd.qcut(subj_df.DPRIME, 2, labels = ['low', 'high'])
#         subj_df = subj_df[['trial', 'zrpe', 'zpm', 'cat_zrpe', 'cat_zpm']] 

        merged_df = roi_df.merge(subj_df, on=['trials'])
        
        for label in merged_df.label.unique():
            fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)
            sns.lineplot(data=merged_df[merged_df.label==label], x='time', y='smoothed_power', 
                         hue='cat_zrpe', errorbar='se', ax=ax, hue_order=['low', 'high'], legend=None)
            sns.despine()
            ax.set(ylabel = 'power (z)', xlabel = 'time (s)')

            plt.savefig(f'{plot_dir}/{event}_{boi}_RPE_{subj_id}_{label}.pdf', dpi=300)
#             plt.close()


In [ ]:
elecs_in_roi

In [ ]:
freq_avged_data = np.expand_dims(freq_avged_data, axis=1)

In [ ]:
freq_avged_data.shape

In [ ]:
elecs_in_roi

In [ ]:
smoothed_arr.shape[1]

In [ ]:
# for region in rois: 
#     # if region == 'TC':
#     #     region = (('Temporal') | ('Temporal Pole') | ('STS'))
#     fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300, sharey=True, sharex=True)
#     for ix, cond in enumerate(conditions): 
#         # Set the times for 
#         n_elecs = np.squeeze(np.stack(tfr_group_data[region][cond])).shape[0]
#         times = np.linspace(0, 1.5, np.squeeze(np.stack(tfr_group_data[region][cond])).shape[2])

# #         if cond_name == 'SME':
# #             plot_data = np.nanmedian(np.nanmedian(np.squeeze(np.stack(tfr_group_data[region][cond])), axis=0), axis=0)
# #         else:
#         plot_data = np.nanmean(np.squeeze(np.stack(tfr_group_data[region][cond])), axis=0)
#         ax.plot(times, np.nanmean(plot_data[freqs<9, :], axis=0))
      
#         ax.set(xlabel='Time (s)', ylabel='Power (z)', title=f'{cond}_{region}_nelec={n_elecs}')
#         # fig.colorbar(im, ax=ax[ix])
#         plt.savefig(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/feedback_theta_{cond_name}_{region}.pdf', dpi=300)


## Second, theta decreases for successful encoding of positive RPEs

In [ ]:
roi = 'OFC'
boi = 'theta'
foi = [2, 8]
freqs = np.logspace(*np.log10([2, 200]), num=30)

In [ ]:
# Let's load the power epochs for specific conditions: 

analysis_evs = ['feedback_start']

# tfr_group_data = {f'{a}': {f'{b}': [] for b in conditions} for a in rois}

for subj_id in tqdm(subj_ids): 
    for event in analysis_evs:
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        
        # Load the data 
        full_tfr = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
        
        # Get electrode df 
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
        
        
        # 1) Immediately crop this to the roi: 
        elecs_in_roi = elec_df[elec_df.salman_region==roi].label.values.tolist()
        if len(elecs_in_roi) == 0:
            print(f'no electrodes in this region in {subj_id}')
            continue
            
        full_tfr.pick(elecs_in_roi)
        
        # 2) Immediately restrict this to the frequency of interest and average it within band 
        freqs_in_boi = (freqs>foi[0]) & (freqs<foi[-1])
        freq_avged_data = np.squeeze(full_tfr._data[:, :, freqs_in_boi, :].mean(axis=2))
        
#         # 3) Smooth data, if need be:
#         if boi == 'hfa':
        # Assuming arr is your numpy array of shape (80, 3, 751)
        smoothed_arr = np.zeros_like(freq_avged_data)
        if len(elecs_in_roi) == 1:
            smoothed_arr[trial, :] = pd.Series(freq_avged_data[trial, :]).rolling(window=foi[-1], min_periods=1, step=1, closed='both').apply(lambda x: np.nanmean(x)).values
        else:
            for trial in range(smoothed_arr.shape[0]):
                for chan in range(smoothed_arr.shape[1]):
                    smoothed_arr[trial, chan, :] = pd.Series(freq_avged_data[trial, chan, :]).rolling(window=foi[-1], min_periods=1, step=1, closed='both').apply(lambda x: np.nanmean(x)).values
#         else:
#             smoothed_arr = freq_avged_data
            
        
        # 4) Arrange this data into a pandas dataframe 
        
        # Reshape the array into a shape that can be converted into DataFrame
        roi_df = [] 
        
        for ix, label in enumerate(elecs_in_roi):
#             if boi == 'hfa':
            if len(elecs_in_roi) == 1:
                flat_pow = smoothed_arr.flatten()
            else:
                flat_pow = smoothed_arr[:,ix,:].flatten()

#             else:
#                 if len(elecs_in_roi) == 1:
#                     flat_pow = freq_avged_data.flatten()
#                 else:
#                     flat_pow = freq_avged_data[:,ix,:].flatten()
            trial_col = np.repeat(np.arange(freq_avged_data.shape[0]) + 1, flat_pow.shape[0]/freq_avged_data.shape[0])
            times_col = list(full_tfr.times) * freq_avged_data.shape[0]

            # Create DataFrame
            df = pd.DataFrame({
                'trial': trial_col,
                'label': label,
                'time': times_col,
                'smoothed_power': flat_pow,
                'band': boi,
                'region': roi
            })
            
            # Smooth data, if need be:
            
            roi_df.append(df)

        roi_df = pd.concat(roi_df).reset_index(drop=True)
            
# # #         # Filter out the nans
# # #         df.dropna(inplace=True)
                    
# # #         # 5) Merge with the appropriate behavioral dataframe (depending on condition)
        trials_to_include = combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_gamble').reset_index(drop=True).trials_gamble.values.astype(int)

        roi_df = roi_df[roi_df['trial'].isin(trials_to_include)]
        
        # 
        subj_df = combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_gamble').reset_index(drop=True)
        subj_df.rename(columns={'trials_gamble':'trial'}, inplace=True)
        subj_df['cat_zrpe'] = pd.qcut(subj_df.zrpe, 2, labels = ['low', 'high'])
        subj_df['cat_zpm'] = pd.qcut(subj_df.zpm, 2, labels = ['low', 'high'])
        subj_df = subj_df[['trial', 'zrpe', 'zpm', 'cat_zrpe', 'cat_zpm', 'hits']] 

        merged_df = roi_df.merge(subj_df, on=['trial'])
        
        for label in merged_df.label.unique():
            fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)
            try:
                sns.lineplot(data=merged_df[(merged_df.label==label) & (merged_df.cat_zrpe=='high')], x='time', y='smoothed_power', 
                             hue='hits', errorbar='se', ax=ax, hue_order=[False, True], legend=None)
            except KeyError:
                print('no data after dropping nans')
                continue
            sns.despine()
            ax.set(ylabel = 'power (z)', xlabel = 'time (s)')

            plt.savefig(f'{plot_dir}/{event}_{roi}_{boi}_RPE_{subj_id}_{label}.pdf', dpi=300)
            plt.close()


Why is the theta plot so jagged? It's because of all the nan's from my absurdity threshold... let's get rid of this it makes me a little bit uncomfortable overall.

In [ ]:
np.where(~np.isnan(smoothed_arr[:, 1, :]).any(axis=1))[0]+1

In [ ]:
smoothed_arr[16, 1, :]

In [ ]:
test_arr = np.squeeze(smoothed_arr[np.where(~np.isnan(smoothed_arr[:, 2, :]).any(axis=1)), 2, :])

In [ ]:
plt.plot(np.nanmean(smoothed_arr[:, 2, :], axis=0), 'k')
plt.plot(np.nanmean(test_arr, axis=0), 'r')

# Plot examples from TFR data

In [ ]:
# plot_data = np.nanmean(zpow, axis=0)

event = 'feedback_start'
subj_id = 'MS036'
day = 'Day1'
zpow = mne.time_frequency.read_tfrs(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR/{event}-tfr.h5')
elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')




for elec in elec_df[elec_df.salman_region=='OFC'].label.iloc[-3:-2]:
       zpow_chan = zpow.copy().pick([elec])._data

       fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)
       times = np.linspace(0, 1.5, np.squeeze(zpow_chan.mean(axis=0)).shape[-1])
       freqs = np.logspace(*np.log10([2, 200]), num=30)


       im = ax.imshow(np.squeeze(np.nanmean(zpow_chan, axis=0)),
                     aspect='auto', origin='lower', cmap='viridis', interpolation = 'bicubic', vmin=-1, vmax=1)            
       fig.colorbar(im, ax=ax)
       freq_res = 5
       time_res = 250 
       ax.set(yticks=np.arange(0, len(freqs))[::freq_res], 
              yticklabels=freqs[::freq_res].astype(int), 
              xticks = np.arange(len(times))[::time_res],
              xticklabels= zpow.times[::time_res])
       
       # Plot a white box around the 70-200 Hz frequencies
       ax.plot([0, len(times)-1], [np.where(freqs>70)[0][0], np.where(freqs>70)[0][0]], 'w-', lw=2)

       plt.savefig(f'{plot_dir}/example_OFC_TFR_MS036_{elec}.pdf', dpi=300)
